The purpose of this notebook is to perform an initial analysis of the collected data on edge ML models. We will focus on visualizing the trends in model sizes over the years. Most of the analyses here are just for forming intuitions and guiding further research questions (and also to present to the December 10 meeting).


In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update(
    {
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "legend.fontsize": 12,
        "font.size": 12,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

In [ ]:
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
df = load_dataset(
    "ljvmiranda921/edgeml-ltl-survey-annotations",
    split="train",
    revision="6dae73653bd3397fedc3af0146451a3401059ae7",
).to_pandas()
# Only get those that were marked as relevant
relevance = 4
df = df[df["relevance_score"] >= relevance].reset_index(drop=True)
print(f"Number of relevant entries: {len(df)} (relevance >= {relevance})")

In [ ]:
# Export to CSV to double-check annotations
df.to_csv(DATA_DIR / "edgeml-survey-entries.csv", index=False)

In [ ]:
df.columns

**Research Question**: Did our definition of small (in terms of model parameter size) changed over time? Let's see the distribution of model sizes over the years. To do so:

In [ ]:
# Did our definition of small (in terms of model parameter size) change over time?
df_plot = df.explode("model_sizes").reset_index(drop=True)
df_plot = df_plot[df_plot["model_sizes"].notna()].copy()
df_plot["model_sizes"] = pd.to_numeric(df_plot["model_sizes"])

yearly_median = df_plot.groupby("year")["model_sizes"].median().reset_index()
yearly_avg = df_plot.groupby("year")["model_sizes"].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    df_plot["year"], df_plot["model_sizes"], alpha=0.5, s=50, label="Individual models"
)
ax.plot(
    yearly_median["year"],
    yearly_median["model_sizes"],
    color="red",
    linewidth=2,
    marker="o",
    label="Yearly Median",
)
# ax.plot(
#    yearly_avg["year"],
#    yearly_avg["model_sizes"],
#    color="blue",
#    linewidth=2,
#    marker="o",
#    label="Yearly Average",
# )

ax.set_xlabel("Year")
ax.set_ylabel("Model Size (Billion Parameters)")
ax.set_title("Model Parameter Size (B) from 2016 to 2025", fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / "model_size_over_time.png", dpi=300)
plt.show()

**Research Question**: Is efficient NLP still English-centric?

In [ ]:
# Is edge ML work English-centric?
import langcodes

df_lang = df.explode("languages_supported").reset_index(drop=True)
df_lang = df_lang[df_lang["languages_supported"].notna()].copy()

programming_languages = {"C++", "Java", "Python", "JavaScript", "C#", "Ruby", "Go"}


def is_valid_language(code):
    if code in programming_languages:
        return False
    try:
        langcodes.Language.get(code)
        return True
    except:
        return False


def get_language_family(code):
    try:
        # fmt: off
        if code in ["en", "de", "fr", "es", "it", "pt", "ru", "hi", "bn", "fa", "el", "pl", "cs", "sv", "no", "da", "nl", "ro", "uk", "bg", "sr", "sl", "sk", "lb", "lt", "lv"]:
            return "Indo-European"
        # fmt: on
        elif code in ["zh"]:
            return "Sino-Tibetan"
        elif code in ["ar", "he"]:
            return "Afroasiatic"
        elif code in ["ja"]:
            return "Japonic"
        elif code in ["ko"]:
            return "Koreanic"
        elif code in ["tr"]:
            return "Turkic"
        elif code in ["fi", "et", "hu"]:
            return "Uralic"
        elif code in ["id", "ms", "tl"]:
            return "Austronesian"
        elif code in ["vi"]:
            return "Austroasiatic"
        elif code in ["ta"]:
            return "Dravidian"
        elif code in ["th"]:
            return "Kra-Dai"
        elif code in ["sw"]:
            return "Niger-Congo"
        else:
            return "Other"
    except:
        return "Other"


df_lang = df_lang[df_lang["languages_supported"].apply(is_valid_language)]
df_lang["language_family"] = df_lang["languages_supported"].apply(get_language_family)
lang_counts = df_lang["languages_supported"].value_counts()

family_colors = {
    "Indo-European": "#1f77b4",
    "Sino-Tibetan": "#ff7f0e",
    "Afroasiatic": "#2ca02c",
    "Austronesian": "#d62728",
    "Austroasiatic": "#9467bd",
    "Dravidian": "#8c564b",
    "Kra-Dai": "#e377c2",
    "Niger-Congo": "#7f7f7f",
    "Japonic": "#bcbd22",
    "Koreanic": "#17becf",
    "Turkic": "#aec7e8",
    "Uralic": "#ffbb78",
    "Other": "#c7c7c7",
}

colors = [family_colors[get_language_family(lang)] for lang in lang_counts.index]

fig, ax = plt.subplots(figsize=(12, 6))
lang_counts.plot(kind="bar", ax=ax, color=colors)
ax.set_xlabel("Language (ISO-639-1)")
ax.set_ylabel("Count")
ax.set_title("Language Distribution in Edge ML Research", fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--", axis="y")
plt.xticks(rotation=45, ha="right")

from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=color, label=family)
    for family, color in family_colors.items()
    if any(get_language_family(lang) == family for lang in lang_counts.index)
]
ax.legend(
    handles=legend_elements,
    title="Language Family\n(based on Ethnologue)",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
)

plt.tight_layout()
plt.savefig(DATA_DIR / "language_distribution.png", dpi=300)
plt.show()

In [ ]:
# Language family distribution
family_counts = df_lang["language_family"].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
family_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Language Family")
ax.set_ylabel("Count")
# ax.set_title("Language Family Distribution in Edge ML Research", fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--", axis="y")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(DATA_DIR / "language_family_distribution.png", dpi=300)
plt.show()

**Research Question:** How does the domain inform the deployment environment for edge ML models? Is there a relationship between where they are deployed (mobile devices, IoT, etc.) and the application domain (healthcare, finance, etc.)?

In [ ]:
import itertools

rows = []
for idx, row in df.iterrows():
    domains = row["application_domain"]
    platforms = row["deployment_platforms"]

    if domains is None or (isinstance(domains, float) and pd.isna(domains)):
        domains = []
    if platforms is None or (isinstance(platforms, float) and pd.isna(platforms)):
        platforms = []

    for domain, platform in itertools.product(domains, platforms):
        rows.append({"application_domain": domain, "deployment_platform": platform})

df_cross = pd.DataFrame(rows)
crosstab = pd.crosstab(df_cross["application_domain"], df_cross["deployment_platform"])
crosstab = crosstab.loc[crosstab.sum(axis=1).sort_values(ascending=False).index]
print(f"\nCross-tabulation shape: {crosstab.shape}")

Where are edgeML models being deployed and in which use-case?

In [ ]:
import numpy as np

crosstab_filtered = crosstab.copy()

if "Other" in crosstab_filtered.index:
    crosstab_filtered = crosstab_filtered.drop("Other", axis=0)
if "Not Specified" in crosstab_filtered.columns:
    crosstab_filtered = crosstab_filtered.drop("Not Specified", axis=1)
# if "Cloud" in crosstab_filtered.columns:
#    crosstab_filtered = crosstab_filtered.drop("Cloud", axis=1)
crosstab_filtered = crosstab_filtered.rename(
    index={"Environmental Science": "Env. Science"},
)

crosstab_pct = crosstab_filtered.div(crosstab_filtered.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(6, 6))

im = ax.imshow(crosstab_pct.values, cmap="Blues", aspect="auto", vmin=0, vmax=100)
# cbar = plt.colorbar(im, ax=ax)
# cbar.set_label("Percentage (%)", rotation=270, labelpad=15)

for i in range(len(crosstab_pct.index)):
    for j in range(len(crosstab_pct.columns)):
        text = ax.text(
            j,
            i,
            f"{crosstab_pct.values[i, j]:.1f}%",
            ha="center",
            va="center",
            color="black",
            fontsize=12,
        )

ax.set_xticks(np.arange(len(crosstab_pct.columns)))
ax.set_yticks(np.arange(len(crosstab_pct.index)))
ax.set_xticklabels(crosstab_pct.columns, rotation=45, ha="right")
ax.set_yticklabels(crosstab_pct.index)
ax.set_xlabel("Deployment Platform")
ax.set_ylabel("Application Domain")
# ax.set_title("Application Domain vs. Deployment Platform (%)", fontweight="bold")

plt.tight_layout()
plt.savefig(DATA_DIR / "application_domain_vs_deployment_platform.png", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

crosstab_pct.plot(kind="barh", stacked=True, ax=ax, colormap="tab10", width=0.8)
ax.set_xlabel("Percentage (%)")
ax.set_ylabel("Application Domain")
# ax.set_title("Deployment Platform Distribution by Application Domain", fontweight="bold")  # fmt: skip
ax.legend(title="Deployment Platform", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(True, alpha=0.3, linestyle="--", axis="x")
ax.set_xlim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
def get_application_domains(application: str, deployment: str):
    deployment_df = df[
        df["deployment_platforms"].apply(lambda x: True if deployment in x else False)
    ].reset_index(drop=True)
    app_df = deployment_df[
        deployment_df["application_domain"].apply(
            lambda x: True if application in x else False
        )
    ].reset_index(drop=True)
    return app_df

In [ ]:
get_application_domains("Environmental Science", "Cloud")

**Research Question**: does multilinguality (non-English) affect the choice of model intervention?

In [ ]:
def categorize_language(languages_supported):
    """Categorize papers as English-only, non-English, or multilingual"""
    valid_langs = [
        lang for lang in languages_supported if lang not in programming_languages
    ]

    if not valid_langs:
        return "Unspecified"

    has_english = "en" in valid_langs
    has_non_english = any(lang != "en" for lang in valid_langs)

    if has_english and not has_non_english:
        return "English-only"
    elif has_non_english and not has_english:
        return "Multilingual"
    elif has_english and has_non_english:
        return "Multilingual"
    else:
        return "Unspecified"


df["language_category"] = df["languages_supported"].apply(categorize_language)

# Check the distribution
print("Language category distribution:")
print(df["language_category"].value_counts())
print()

# Now let's explode the methods to analyze technique usage
# We'll focus on papers that explicitly mention English or non-English
df_methods = df[
    df["language_category"].isin(["English-only", "Non-English", "Multilingual"])
].copy()
df_methods_exploded = df_methods.explode("methods_used").reset_index(drop=True)
df_methods_exploded = df_methods_exploded[
    df_methods_exploded["methods_used"].notna()
].copy()

print(f"Total papers with language info and methods: {len(df_methods)}")
print(f"Total method-language pairs: {len(df_methods_exploded)}")
print()

# Create a crosstab of techniques vs language categories
technique_lang_counts = pd.crosstab(
    df_methods_exploded["methods_used"], df_methods_exploded["language_category"]
)

print("Technique usage by language category:")
print(technique_lang_counts)
print()

In [ ]:
# Create a grouped bar chart comparing English-only vs Non-English techniques
# For clarity, we'll combine Multilingual with Non-English

# Create combined categories for clearer comparison
df_methods_exploded["lang_group"] = df_methods_exploded["language_category"].apply(
    lambda x: (
        "Non-English (incl. Multilingual)"
        if x in ["Non-English", "Multilingual"]
        else "English-only"
    )
)

# Create crosstab with combined categories
technique_comparison = pd.crosstab(
    df_methods_exploded["methods_used"], df_methods_exploded["lang_group"]
)

# Sort by total usage
technique_comparison["Total"] = technique_comparison.sum(axis=1)
technique_comparison = technique_comparison.sort_values("Total", ascending=True)
technique_comparison = technique_comparison.drop("Total", axis=1)

# Create a grouped bar chart
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(technique_comparison.index))
width = 0.35

bars1 = ax.barh(
    x - width / 2,
    technique_comparison["English-only"],
    width,
    label="English-only",
    color="steelblue",
    alpha=0.8,
)
bars2 = ax.barh(
    x + width / 2,
    technique_comparison["Non-English (incl. Multilingual)"],
    width,
    label="Multilingual",
    color="coral",
    alpha=0.8,
)

ax.set_xlabel("Number of Papers")
ax.set_yticks(x)
ax.set_yticklabels(technique_comparison.index)
ax.legend()
ax.grid(True, alpha=0.3, linestyle="--", axis="x")

plt.tight_layout()
plt.savefig(DATA_DIR / "technique_comparison_language.png", dpi=300)
plt.show()

print("\nAbsolute counts:")
print(technique_comparison)

**Research Question**: does multilinguality (non-English) affect the choice of NLP application?

In [ ]:
# Subject area distribution (percentage)
df_subject = df.explode("subject_areas").reset_index(drop=True)
df_subject = df_subject[df_subject["subject_areas"].notna()].copy()

subject_counts = df_subject["subject_areas"].value_counts()
subject_pct = (subject_counts / len(df) * 100).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
subject_pct.plot(kind="barh", ax=ax, color="steelblue", alpha=0.8)
ax.set_xlabel("Percentage of Papers (%)")
ax.set_ylabel("")
# ax.set_title("Distribution of Subject Areas in Edge ML Research", fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--", axis="x")

# Add percentage labels on the bars
for i, v in enumerate(subject_pct):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=10)

plt.tight_layout()
plt.savefig(DATA_DIR / "subject_area_distribution.png", dpi=300)
plt.show()

print(f"\nTotal papers: {len(df)}")
print(f"Papers with subject areas: {subject_counts.sum()}")
print("\nSubject area counts:")
print(subject_counts)

In [ ]:
# Compare subject areas: Multilingual vs English-only
df_subject_lang = df.explode("subject_areas").reset_index(drop=True)
df_subject_lang = df_subject_lang[df_subject_lang["subject_areas"].notna()].copy()

# Filter for only English-only and Multilingual papers
df_subject_lang = df_subject_lang[
    df_subject_lang["language_category"].isin(["English-only", "Multilingual"])
].copy()

# Create crosstab of subject areas vs language categories
subject_lang_crosstab = pd.crosstab(
    df_subject_lang["subject_areas"], df_subject_lang["language_category"]
)

# Calculate percentages (percentage of papers in each language category that have this subject area)
total_papers_by_lang = df[
    df["language_category"].isin(["English-only", "Multilingual"])
]["language_category"].value_counts()

subject_lang_pct = subject_lang_crosstab.copy()
for col in subject_lang_pct.columns:
    subject_lang_pct[col] = (subject_lang_pct[col] / total_papers_by_lang[col]) * 100

# Calculate the difference (Multilingual % - English-only %)
subject_lang_pct["Difference"] = (
    subject_lang_pct["Multilingual"] - subject_lang_pct["English-only"]
)
subject_lang_pct_sorted = subject_lang_pct.sort_values("Difference", ascending=True)

# Create diverging bar chart showing the difference
fig, ax = plt.subplots(figsize=(10, 7))

colors = [
    "coral" if x > 0 else "steelblue" for x in subject_lang_pct_sorted["Difference"]
]
bars = ax.barh(
    subject_lang_pct_sorted.index,
    subject_lang_pct_sorted["Difference"],
    color=colors,
    alpha=0.8,
)

ax.axvline(x=0, color="black", linewidth=0.8, linestyle="-")
ax.set_xlabel("Difference in Percentage Points\n(Multilingual % - English-only %)")
ax.set_ylabel("Subject Area")
ax.set_title(
    "Subject Area Prevalence: Multilingual vs English-only Papers", fontweight="bold"
)
ax.grid(True, alpha=0.3, linestyle="--", axis="x")

# Add a legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="coral", alpha=0.8, label="More common in Multilingual"),
    Patch(facecolor="steelblue", alpha=0.8, label="More common in English-only"),
]
ax.legend(handles=legend_elements, loc="best")

plt.tight_layout()
plt.savefig(DATA_DIR / "subject_area_language_difference.png", dpi=300)
plt.show()

print(f"\nEnglish-only papers: {total_papers_by_lang['English-only']}")
print(f"Multilingual papers: {total_papers_by_lang['Multilingual']}")
print("\nPercentage difference (Multilingual - English-only):")
print(subject_lang_pct_sorted[["English-only", "Multilingual", "Difference"]].round(1))

**Balanced comparison**: Sample 35-40 papers from English-only to match Multilingual sample size

In [ ]:
# Sample 35-40 papers from English-only to match Multilingual dataset size
import numpy as np

# Set random seed for reproducibility
# np.random.seed(21)

# Get English-only and Multilingual papers
english_only_papers = df[df["language_category"] == "English-only"]
multilingual_papers = df[df["language_category"] == "Multilingual"]

print(f"Original English-only papers: {len(english_only_papers)}")
print(f"Multilingual papers: {len(multilingual_papers)}")

# Sample 35 papers from English-only to match multilingual size
sample_size = len(multilingual_papers)  # 35 papers
sample_size = 100
english_only_sampled = english_only_papers.sample(n=sample_size)

print(f"\nSampled English-only papers: {len(english_only_sampled)}")

# Combine sampled English-only with all Multilingual papers
df_balanced = pd.concat([english_only_sampled, multilingual_papers]).reset_index(
    drop=True
)

print(f"Total balanced dataset: {len(df_balanced)}")
print(f"\nLanguage category distribution in balanced dataset:")
print(df_balanced["language_category"].value_counts())

# Regenerate the subject area comparison with balanced dataset
df_subject_lang_balanced = df_balanced.explode("subject_areas").reset_index(drop=True)
df_subject_lang_balanced = df_subject_lang_balanced[
    df_subject_lang_balanced["subject_areas"].notna()
].copy()

# Filter for only English-only and Multilingual papers
df_subject_lang_balanced = df_subject_lang_balanced[
    df_subject_lang_balanced["language_category"].isin(["English-only", "Multilingual"])
].copy()

# Create crosstab of subject areas vs language categories
subject_lang_crosstab_balanced = pd.crosstab(
    df_subject_lang_balanced["subject_areas"],
    df_subject_lang_balanced["language_category"],
)

# Calculate percentages (percentage of papers in each language category that have this subject area)
total_papers_by_lang_balanced = df_balanced[
    df_balanced["language_category"].isin(["English-only", "Multilingual"])
]["language_category"].value_counts()

subject_lang_pct_balanced = subject_lang_crosstab_balanced.copy()
for col in subject_lang_pct_balanced.columns:
    subject_lang_pct_balanced[col] = (
        subject_lang_pct_balanced[col] / total_papers_by_lang_balanced[col]
    ) * 100

# Calculate the difference (Multilingual % - English-only %)
subject_lang_pct_balanced["Difference"] = (
    subject_lang_pct_balanced["Multilingual"]
    - subject_lang_pct_balanced["English-only"]
)
subject_lang_pct_sorted_balanced = subject_lang_pct_balanced.sort_values(
    "Difference", ascending=True
)

# Create diverging bar chart showing the difference
fig, ax = plt.subplots(figsize=(12, 7))

colors = [
    "coral" if x > 0 else "steelblue"
    for x in subject_lang_pct_sorted_balanced["Difference"]
]
bars = ax.barh(
    subject_lang_pct_sorted_balanced.index,
    subject_lang_pct_sorted_balanced["Difference"],
    color=colors,
    alpha=0.8,
)

ax.axvline(x=0, color="black", linewidth=0.8, linestyle="-")
ax.set_xlabel("Difference in Percentage Points\n(Multilingual % - English-only %)")
ax.set_ylabel("Subject Area")
# ax.set_title(
#    f"Subject Area Prevalence: Multilingual vs English-only Papers",
#    fontweight="bold",
# )
ax.grid(True, alpha=0.3, linestyle="--", axis="x")

# Add a legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="coral", alpha=0.8, label="More common in Multilingual"),
    Patch(facecolor="steelblue", alpha=0.8, label="More common in English-only"),
]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.savefig(DATA_DIR / "subject_area_language_difference_balanced.png", dpi=300)
plt.show()

print(
    f"\nEnglish-only papers (sampled): {total_papers_by_lang_balanced['English-only']}"
)
print(f"Multilingual papers: {total_papers_by_lang_balanced['Multilingual']}")
print("\nPercentage difference (Multilingual - English-only):")
print(
    subject_lang_pct_sorted_balanced[
        ["English-only", "Multilingual", "Difference"]
    ].round(1)
)

In [ ]:
df[df["methods_used"].apply(lambda x: "Data-Efficient Training" in x)].sort_values(
    "year", ascending=False
).head(10)